**Project:** E-Commerce Analytics Dashboard
## Mục tiêu
1. Đọc dữ liệu thô và khảo sát chất lượng dữ liệu
2. Tách các đơn hủy ra một dataset riêng để phân tích sau
3. Làm sạch các giao dịch bị lỗi
4. Tạo cột phái sinh
5. Xuất ra 3 dataset sạch

## 1. Setup môi trường

In [2]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns
import warnings

warnings.filterwarnings('ignore')

#Cấu hình hiển thị sao cho đẹp
pd.set_option('display.max_columns', None)
pd.set_option('display.float_format', '{:,.2f}'.format)

#Cấu hình style cho biểu đồ
sns.set_style('whitegrid')
plt.rcParams['figure.figsize'] = (12, 5)

print('Đã import xong các thư viện')

Đã import xong các thư viện


## 2. Đọc dữ liệu thô

In [4]:
#Đọc dữ liệu
df = pd.read_csv('../data/raw/E-Commerce Data.csv', encoding='unicode_escape')

#Xem 5 dòng đầu
df.head()

,InvoiceNo,StockCode,Description,Quantity,InvoiceDate,UnitPrice,CustomerID,Country
0,536365,85123A,WHITE HANGING HEART T-LIGHT HOLDER,6,12/1/2010 8:26,2.55,"17,850.00",United Kingdom
1,536365,71053,WHITE METAL LANTERN,6,12/1/2010 8:26,3.39,"17,850.00",United Kingdom
2,536365,84406B,CREAM CUPID HEARTS COAT HANGER,8,12/1/2010 8:26,2.75,"17,850.00",United Kingdom
3,536365,84029G,KNITTED UNION FLAG HOT WATER BOTTLE,6,12/1/2010 8:26,3.39,"17,850.00",United Kingdom
4,536365,84029E,RED WOOLLY HOTTIE WHITE HEART.,6,12/1/2010 8:26,3.39,"17,850.00",United Kingdom


## 3. Khảo sát chất lượng dữ liệu

In [5]:
print(f'Số dòng: {df.shape[0]:,}')
print(f'Số cột: {df.shape[1]}')

Số dòng: 541,909
Số cột: 8


### 3.1. Kiểu dữ liệu của mỗi cột

In [6]:
df.info()

<class 'pandas.DataFrame'>
RangeIndex: 541909 entries, 0 to 541908
Data columns (total 8 columns):
 #   Column       Non-Null Count   Dtype  
---  ------       --------------   -----  
 0   InvoiceNo    541909 non-null  str    
 1   StockCode    541909 non-null  str    
 2   Description  540455 non-null  str    
 3   Quantity     541909 non-null  int64  
 4   InvoiceDate  541909 non-null  str    
 5   UnitPrice    541909 non-null  float64
 6   CustomerID   406829 non-null  float64
 7   Country      541909 non-null  str    
dtypes: float64(2), int64(1), str(5)
memory usage: 33.1 MB


### 3.2. Thống kê

In [7]:
df.describe()

,Quantity,UnitPrice,CustomerID
count,"541,909.00","541,909.00","406,829.00"
mean,9.55,4.61,"15,287.69"
std,218.08,96.76,"1,713.60"
min,"-80,995.00","-11,062.06","12,346.00"
25%,1.00,1.25,"13,953.00"
50%,3.00,2.08,"15,152.00"
75%,10.00,4.13,"16,791.00"
max,"80,995.00","38,970.00","18,287.00"


### 3.3. Dữ liệu bị mất/thiếu

In [8]:
missing = df.isnull().sum()
missing_pct = (missing / len(df) * 100). round(2)

missing_df = pd.DataFrame({
    'Số dòng thiếu': missing,
    'Tỷ lệ (%)': missing_pct
})

missing_df[missing_df['Số dòng thiếu'] > 0]

,Số dòng thiếu,Tỷ lệ (%)
Description,1454,0.27
CustomerID,135080,24.93


### 3.4. Kiểm tra dòng trùng lặp

In [9]:
n_duplicates = df.duplicated().sum()
print(f'Số dòng trùng lặp hoàn toàn: {n_duplicates:,}')
print(f'Tỷ lệ: {n_duplicates/len(df)*100:.2f}%')

Số dòng trùng lặp hoàn toàn: 5,268
Tỷ lệ: 0.97%


## 4. Điều tra chi tiết về vấn đề còn tồn tại

### 4.1. Kiểm tra tại sao Quantity lại bị âm